# ABSA Agent Pipeline — Google Colab

This notebook runs the **Aspect-Based Sentiment Analysis (ABSA)** multi-agent pipeline on Google Colab.

The pipeline processes restaurant customer reviews through **6 sequential stages**:
1. **Toxicity Detection** — Screens for toxic/abusive content
2. **Aspect Extraction** — Identifies restaurant experience aspects
3. **Sentiment Analysis** — Classifies sentiment per aspect
4. **Policy Evaluation & Escalation** — Checks thresholds and creates tickets
5. **Response Generation** — Composes customer-facing response
6. **Output Guardrail** — Validates response before publishing

**Conditional Flow Control:**
- Toxic reviews are terminated at Stage 1 — stages 2-6 are skipped entirely
- Guardrail failures (REVISE / ESCALATE_TO_HUMAN) are written to a human review queue

---

## 1. Clone Repository & Install Dependencies

In [ ]:
!git clone https://github.com/manaranjanp/absa_sdk /content/absa

In [ ]:
%cd /content/absa

In [ ]:
!pip install -q google-adk>=1.2.0 litellm>=1.40.0 python-dotenv>=1.0.0 langsmith>=0.1.0

## 2. Configure API Key

Enter your Groq API key when prompted. Get one at https://console.groq.com/keys

In [ ]:
import os
from getpass import getpass

os.environ['GROQ_API_KEY'] = getpass('Enter your GROQ_API_KEY: ')

MODEL_NAME = 'groq/llama-3.3-70b-versatile'

print(f'Model: {MODEL_NAME}')

## 3. Setup Observability & Create Pipeline

In [ ]:
import json
import time
import random

from observability import setup_observability, setup_litellm_logging
setup_observability()
setup_litellm_logging(verbose=False)

from pipeline.absa_pipeline import create_absa_pipeline, run_pipeline
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner

print(f'Using model: {MODEL_NAME}')

# Create pipeline and shared runner (reused across reviews)
pipeline = create_absa_pipeline(MODEL_NAME)
session_service = InMemorySessionService()
runner = Runner(
    agent=pipeline,
    app_name='absa_pipeline_app',
    session_service=session_service,
)

print(f'Pipeline created with {len(pipeline.sub_agents)} sub-agents:')
for i, agent in enumerate(pipeline.sub_agents):
    cb = 'before_agent_callback' if hasattr(agent, 'before_agent_callback') and agent.before_agent_callback else ''
    acb = 'after_agent_callback' if hasattr(agent, 'after_agent_callback') and agent.after_agent_callback else ''
    callbacks = ', '.join(filter(None, [cb, acb]))
    suffix = f'  ({callbacks})' if callbacks else ''
    print(f'  [{i+1}] {agent.name}{suffix}')

## 4. Load Sample Reviews

In [ ]:
with open('data/sample_reviews.json', 'r') as f:
    all_reviews = json.load(f)

print(f'Loaded {len(all_reviews)} sample reviews\n')
for r in all_reviews[:5]:
    print(f'  {r["review_id"]} | {r["review_text"][:70]}...')
if len(all_reviews) > 5:
    print(f'  ... and {len(all_reviews) - 5} more')

## 5. Run Single Review — Positive (REV-001)

Process a positive multi-aspect review. All 6 stages should execute normally.

In [ ]:
review = all_reviews[0]  # REV-001: Positive multi-aspect review
print(f'Review: {review["review_id"]}')
print(f'Text:   {review["review_text"]}\n')

start = time.time()
result = await run_pipeline(
    review,
    model_name=MODEL_NAME,
    pipeline=pipeline,
    session_service=session_service,
    runner=runner,
)
print(f'Completed in {time.time() - start:.1f}s')

### Inspect Stage Results

In [ ]:
print('=== Toxicity Result ===')
print(json.dumps(result.get('toxicity_result', {}), indent=2))

print('\n=== Aspect Extraction ===')
print(json.dumps(result.get('aspect_result', {}), indent=2))

print('\n=== Sentiment Analysis ===')
print(json.dumps(result.get('sentiment_result', {}), indent=2))

In [ ]:
print('=== Escalation Result ===')
print(json.dumps(result.get('escalation_result', {}), indent=2))

print('\n=== Generated Response ===')
resp = result.get('response_result', {})
print(resp.get('response_text', 'No response generated'))

print('\n=== Guardrail Result ===')
print(json.dumps(result.get('guardrail_result', {}), indent=2))

## 6. Run Single Review — Toxic (REV-004)

Demonstrate conditional flow: toxic reviews are terminated at Stage 1.
The `before_agent_callback` on stages 2-6 checks `toxicity_result` and
skips remaining agents — no LLM calls are made for stages 2-6.

In [ ]:
toxic_review = all_reviews[3]  # REV-004: Toxic review
print(f'Review: {toxic_review["review_id"]}')
print(f'Text:   {toxic_review["review_text"]}\n')

start = time.time()
toxic_result = await run_pipeline(
    toxic_review,
    model_name=MODEL_NAME,
    pipeline=pipeline,
    session_service=session_service,
    runner=runner,
)
print(f'Completed in {time.time() - start:.1f}s')

In [ ]:
# Verify stages 2-6 were skipped (results should be empty dicts)
print('Toxicity:', json.dumps(toxic_result.get('toxicity_result', {}), indent=2))
print(f'\nAspect result:     {toxic_result.get("aspect_result", {})}')
print(f'Sentiment result:  {toxic_result.get("sentiment_result", {})}')
print(f'Escalation result: {toxic_result.get("escalation_result", {})}')
print(f'Response result:   {toxic_result.get("response_result", {})}')
print(f'Guardrail result:  {toxic_result.get("guardrail_result", {})}')

## 7. Run Single Review — Negative with Escalation (REV-009)

Process a negative review to trigger escalation.

In [ ]:
esc_review = all_reviews[8]  # REV-009: Negative review
print(f'Review: {esc_review["review_id"]}')
print(f'Text:   {esc_review["review_text"]}\n')

start = time.time()
esc_result = await run_pipeline(
    esc_review,
    model_name=MODEL_NAME,
    pipeline=pipeline,
    session_service=session_service,
    runner=runner,
)
print(f'Completed in {time.time() - start:.1f}s')

## 8. Run Batch — Random Sample

Process a batch of random reviews and print a summary
(mirrors `run_pipeline.py --sample N`).

In [ ]:
N_SAMPLES = 5

batch = random.sample(all_reviews, N_SAMPLES)
print(f'Selected {len(batch)} reviews for batch processing:\n')
for r in batch:
    print(f'  {r["review_id"]} | {r["review_text"][:70]}...')

batch_results = []
total_start = time.time()

for i, review in enumerate(batch):
    print(f'\n{"#"*60}')
    print(f'  Review {i + 1} of {len(batch)}')
    print(f'{"#"*60}')

    start = time.time()
    result = await run_pipeline(
        review,
        model_name=MODEL_NAME,
        pipeline=pipeline,
        session_service=session_service,
        runner=runner,
    )
    print(f'[Pipeline] Review {i + 1} completed in {time.time() - start:.1f}s')
    batch_results.append(result)

total_elapsed = time.time() - total_start

# Batch summary (same as run_pipeline.py)
print(f'\n{"="*60}')
print(f'  BATCH SUMMARY')
print(f'{"="*60}')
print(f'  Total reviews:    {len(batch)}')
print(f'  Total time:       {total_elapsed:.1f}s')
print(f'  Avg per review:   {total_elapsed / len(batch):.1f}s')

toxic_count = sum(1 for r in batch_results if r.get('toxicity_result', {}).get('is_toxic', False))
print(f'  Toxic (terminated): {toxic_count}')
print(f'  Processed:        {len(batch) - toxic_count}')

escalated = sum(1 for r in batch_results if r.get('escalation_result', {}).get('escalation_ticket'))
print(f'  Escalations:      {escalated}')

passed = sum(1 for r in batch_results if r.get('guardrail_result', {}).get('guardrail_passed', False))
print(f'  Guardrail passed: {passed}')
print(f'{"="*60}\n')

## 9. Inspect Escalation Tickets

Check escalation tickets written to `data/escalation_tickets.json`.

In [ ]:
tickets_path = 'data/escalation_tickets.json'
if os.path.exists(tickets_path):
    with open(tickets_path, 'r') as f:
        lines = f.readlines()
    tickets = [json.loads(line) for line in lines if line.strip()]
    print(f'=== Escalation Tickets ({len(tickets)} total) ===')
    for ticket in tickets:
        print(json.dumps(ticket, indent=2))
        print()
else:
    print('No escalation tickets created yet.')

## 10. Inspect Human Review Queue

Check responses that were flagged by the guardrail (REVISE / ESCALATE_TO_HUMAN)
and written to `data/human_review_queue.json` by the `after_agent_callback`.

In [ ]:
human_review_path = 'data/human_review_queue.json'
if os.path.exists(human_review_path):
    with open(human_review_path, 'r') as f:
        lines = f.readlines()
    entries = [json.loads(line) for line in lines if line.strip()]
    print(f'=== Human Review Queue ({len(entries)} entries) ===')
    for entry in entries:
        print(json.dumps(entry, indent=2))
        print()
else:
    print('No responses queued for human review.')

## 11. Inspect Pipeline Results Log

All pipeline summaries are appended to `data/pipeline_results.log`.

In [ ]:
results_path = 'data/pipeline_results.log'
if os.path.exists(results_path):
    with open(results_path, 'r') as f:
        print(f.read())
else:
    print('No pipeline results logged yet.')